In [1]:
import os
import re
import ast

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.embeddings import BaseEmbedding
from llama_index.llms.groq import Groq
from langchain_community.vectorstores import Chroma
from chromadb.utils import embedding_functions
from llama_index.core import VectorStoreIndex, Settings
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
import glob
from langchain_core.documents import Document
from dotenv import load_dotenv
import json
import networkx as nx
from operator import itemgetter
import math
from collections import defaultdict
from sklearn.cluster import KMeans
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from llama_index.core.llama_pack import download_llama_pack
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.embeddings import GPT4AllEmbeddings

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

# Replaces OpenAI settings
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = Groq(model="llama-3.3-70b-versatile", api_key="GROQ_API_KEY")

download_llama_pack(
    "SemanticChunkingQueryEnginePack",
    "./semantic_chunking_pack"
    # leave the below line commented out if using the notebook on main
    # llama_hub_url="https://raw.githubusercontent.com/run-llama/llama-hub/jerry/add_semantic_chunker/llama_hub"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


llama_index.packs.node_parser_semantic_chunking.base.SemanticChunkingQueryEnginePack

In [2]:
# Define the path to the directory
dir_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\system_prompts"

# Create a dictionary to store the file contents
system_prompts = {}

# Iterate over all .txt files in the directory
for file_path in glob.glob(os.path.join(dir_path, "*.txt")):
    # Get the base name of the file (without .txt)
    base_name = os.path.basename(file_path)
    file_name = os.path.splitext(base_name)[0]

    # Open the file and read its content
    with open(file_path, 'r', encoding='utf-8-sig') as file:
        content = file.read()

    # Store the content in the dictionary
    system_prompts[file_name] = content
    # Print the name of the file
    print(f"Loaded system prompt: {file_name}")
# Print the keys of the dictionary
print(system_prompts.keys())

# Load graph data from CSV
df_graph = pd.read_csv(r'C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output\graph.csv', sep="|")

Loaded system prompt: prompts
dict_keys(['prompts'])


In [3]:
# function to create a NetworkX graph from JSON data
def load_graph_from_json(json_file_path):
    try:
        with open(json_file_path, 'r', encoding='utf-8-sig') as file:
            json_data = json.load(file)
    except FileNotFoundError:
        print(f"File not found: {json_file_path}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON from the file: {json_file_path}. Error: {e}")
        return None
    except Exception as e:
        print(f"Unexpected error while reading the file: {e}")
        return None

    G = nx.Graph()
    for node in json_data['nodes']:
        G.add_node(node['id'])
    for link in json_data['links']:
        G.add_edge(link['source'], link['target'], weight=link['weight'], title=link['title'])
    return G

### 🏗️ Function: `load_graph_from_json` (Graph Reconstruction)

This function reconstructs a live **NetworkX** graph object from a serialized JSON file, serving as the bridge between static data and active legal reasoning.

#### 🛠 Key Operations:
* **Encoding Resilience**: Utilizes `utf-8-sig` to ensure cross-platform compatibility with Windows-generated text files.
* **Object Mapping**: Iteratively populates the graph with **Nodes** (Legal Entities) and **Weighted Edges** (Relationships).
* **Metadata Retention**: Preserves the `weight` and `title` attributes, which are essential for the **Advanced RAG** engine to prioritize the most relevant evidence.

#### 📍 Practical Application:
It transforms the results of your document extraction into a navigable mathematical structure, allowing the **Junior Law Consultant** to perform complex link analysis and community detection.

In [4]:
# function to load df from json for chunk retrieval based on node and chunk id
def load_chunks_dataframe(json_file_path):
    try:
        with open(json_file_path, 'r', encoding='utf-8-sig') as file:
            data = json.load(file)

        # Extracting chunks and their IDs
        chunks = []
        for link in data['links']:
            chunk_ids = link.get('chunk_id', '').split(',')
            text = link.get('title', '')  # Assuming 'title' contains the text associated with the chunk
            for chunk_id in chunk_ids:
                if chunk_id:
                    chunks.append({'chunk_id': chunk_id, 'text': text})

        return pd.DataFrame(chunks)

    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()  # Return an empty DataFrame in case of an error

### 📖 Function: `load_chunks_dataframe` (Source Evidence Mapping)

This function reconstructs the link between the structured Knowledge Graph and the unstructured source text.

#### 🛠 Key Operations:
* **JSON Flattening**: It expands the comma-separated `chunk_id` strings within graph links into individual row entries.
* **Evidence Pairing**: Maps each chunk identifier to its textual content, typically stored in the relationship `title`.
* **DataFrame Serialization**: Returns a Pandas DataFrame for high-speed indexing during the retrieval phase.

#### 📍 Practical Benefit:
This allows the **Senior Law Consultant** agent to verify any graph-based relationship by instantly retrieving the verbatim text chunks from the `lawsuit.txt` that support the claim.

In [5]:
# function to get top-n contextually closest nodes and edges for a given node
def get_top_contextual_nodes_edges(graph, node, top_n=5):
    if node not in graph:
        return []
    neighbors = [(n, graph[node][n]['weight']) for n in graph.neighbors(node)]
    sorted_neighbors = sorted(neighbors, key=itemgetter(1), reverse=True)
    return sorted_neighbors[:top_n]

### 🔍 Function: `get_top_contextual_nodes_edges` (Neighbor Analysis)

This function performs a localized search within the Knowledge Graph to identify the most relevant structural context for a given entity.

#### 🛠 Logic Breakdown:
* **Validation**: Ensures the requested node exists within the `NetworkX` graph object.
* **Weight-Based Prioritization**: Retrieves all direct neighbors and sorts them by their `weight` attribute (frequency of mention in the source text).
* **Context Truncation**: Limits the output to the top 5 most significant relationships to prevent context-window overflow in the LLM.

#### 📍 Practical Application:
By identifying the strongest structural neighbors, the system can provide "related concept" context. For example, when retrieving info on a **Defendant**, the system automatically pulls in the **Primary Allegation** because they share the highest weighted edge in the graph.

In [6]:
# Load the graph
graph = load_graph_from_json(r'C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output\graph_data.json')
# load the chunk_dataframe
chunks_dataframe = load_chunks_dataframe(r'C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output\graph_data.json')

# ==============================
# Calculate tf-idf-scores for all nodes in context
# ==============================
#region tf-idf logic for nodes context

#Calculate Term Frequency (TF) for nodes context
# Initialize a dictionary to count occurrences
node_context_count = {}
total_contexts = set()

# Iterate over the edges in the NetworkX graph
for (source, target, data) in graph.edges(data=True):
    context = data.get('title', 'No title')
    total_contexts.add(context)

    if source not in node_context_count:
        node_context_count[source] = {}
    if target not in node_context_count:
        node_context_count[target] = {}

    node_context_count[source].setdefault(context, 0)
    node_context_count[target].setdefault(context, 0)
    node_context_count[source][context] += 1
    node_context_count[target][context] += 1

# Calculate Inverse Document Frequency (IDF)
idf_scores = {}
num_contexts = len(total_contexts)

for node, contexts in node_context_count.items():
    idf_scores[node] = math.log(num_contexts / len(contexts))

# Calculate TF-IDF Scores
tf_idf_scores = {}

for node, contexts in node_context_count.items():
    tf_idf_scores[node] = {}
    for context, count in contexts.items():
        tf = count / len(contexts)
        idf = idf_scores[node]
        tf_idf_scores[node][context] = tf * idf
# for custom algo:
# Extract a single TF-IDF score per node (e.g., the maximum score)
single_tf_idf_scores = {node: max(contexts.values()) for node, contexts in tf_idf_scores.items()}

### 📊 Topological TF-IDF Analysis

This block implements a custom statistical weighting engine to prioritize Knowledge Graph entities based on their contextual uniqueness.

#### 🛠 Mathematical Workflow:
* **Contextual TF**: Counts node occurrences across relationship 'titles' to measure local activity.
* **Structural IDF**: Penalizes ubiquitous, generic nodes while boosting rare, specific legal entities.
* **Importance Extraction**: Distills multi-contextual scores into a single `max()` value per node for use in the final retrieval ranking.

#### 📍 Practical Result:
This logic allows the retrieval engine to mathematically "ignore" high-frequency but low-info nodes (like 'Statement' or 'ID') in favor of high-info nodes (like 'Breach of Warranty') when building the context for the LLM.

In [7]:
# function to generate embeddings for the graph
def generate_embeddings(df, column_name):
   
    #texts = df[column_name].tolist()
    texts = df[column_name].astype(str).tolist()  # Ensure all elements are strings
     # Use the model defined in Settings
    embed_model = Settings.embed_model
    embeddings = embed_model.get_text_embedding_batch(texts)
    return embeddings

### 🔢 Function: `generate_embeddings` (Graph Vectorization)

This function transforms the symbolic labels of our Knowledge Graph into a latent vector space.

#### 🛠 Logic Breakdown:
* **Type Enforcement**: Uses `.astype(str)` to prevent serialization errors during the API call.
* **Batch Processing**: Utilizes `embed_documents` to send the entire column at once, which is more efficient than processing nodes one by one.
* **Semantic Mapping**: Converts legal entities and relationships into 1536-dimensional vectors (if using OpenAI) or 384-dimensional vectors (if using BAAI).

#### 📍 Practical Benefit:
This allows the **Junior Law Consultant** to perform "Fuzzy Matching." It can identify that a user's query about a "Contract Breach" is mathematically related to a graph edge labeled "Violation of Agreement."

In [8]:
# 4. Apply to your Graph Dataframe
df_graph['node_1_embedding'] = generate_embeddings(df_graph, 'node_1')
df_graph['node_2_embedding'] = generate_embeddings(df_graph, 'node_2')
df_graph['edge_embedding'] = generate_embeddings(df_graph, 'edge')

print("🚀 Success: Graph fully vectorized using BAAI/bge-small-en-v1.5")

🚀 Success: Graph fully vectorized using BAAI/bge-small-en-v1.5


In [9]:
# ==============================
# topic clustering of indiv nodes
# ==============================

node_embeddings_sum = defaultdict(list)

### 🏷️ Step 1: Entity Vector Aggregation

We initialize a `defaultdict` to aggregate the high-dimensional embeddings for every unique entity found in the Knowledge Graph.

#### 🛠 Technical Mechanics:
* **Collision Handling**: Using a `list` as the default factory allows us to store multiple contextual vectors for a single node (e.g., a 'Defendant' appearing in multiple triplets).
* **Data Preparation**: This structure is the prerequisite for calculating the **Centroid Embedding**, which provides a stable mathematical representation for each legal entity before we perform K-Means clustering.

#### 📍 Practical Benefit:
This ensures that the AI's understanding of a person like **"John Smith"** is informed by *every* relationship he is involved in, rather than just the first time his name appears in the text.

In [10]:
# Iterate over the DataFrame and sum embeddings for each node
for _, row in df_graph.iterrows():
    node_embeddings_sum[row['node_1']].append(row['node_1_embedding'])
    node_embeddings_sum[row['node_2']].append(row['node_2_embedding'])

### 📥 Step 2: Contextual Vector Aggregation

This loop performs the heavy lifting of gathering multi-contextual embeddings for every entity in the Knowledge Graph.

#### 🛠 Technical Mechanics:
* **Row-wise Traversal**: Iterates through the `df_graph` to isolate both subjects (`node_1`) and objects (`node_2`).
* **Multi-Vector Storage**: Leverages the `defaultdict` to store a list of embeddings for each entity. This accounts for the fact that entities appear in multiple different relationship contexts.
* **Feature Preservation**: By capturing the embedding from every relationship, we ensure no contextual nuance is lost before the averaging phase.

#### 📍 Practical Application:
This ensures that if **"Contract Exhibit A"** is linked to both "Financial Claims" and "Missing Deadlines," the final mathematical representation of that exhibit will reflect its importance to both topics.

In [11]:
import numpy as np
from sklearn.cluster import KMeans

# 1. Average the embeddings for each node
# This creates a single "central meaning" vector for each legal entity
node_embeddings_avg = {node: np.mean(embeddings, axis=0) for node, embeddings in node_embeddings_sum.items()}

# 2. Convert embeddings to a list for clustering
nodes, embeddings = zip(*node_embeddings_avg.items())
embeddings_list = list(embeddings)

# 3. 🔥 THE FIX: Calculate n_clusters dynamically
# We want up to 5 clusters, but we must not exceed the number of nodes we actually have.
num_nodes = len(nodes)
n_clusters = min(5, num_nodes) 

print(f"📊 Clustering {num_nodes} nodes into {n_clusters} thematic groups...")

# 4. Perform K-Means clustering only if we have nodes
if n_clusters > 0:
    # n_init=10 is added to ensure stability and compatibility with newer sklearn versions
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    clusters = kmeans.fit_predict(embeddings_list)

    # 5. Map nodes to their clusters
    node_cluster_mapping = dict(zip(nodes, clusters))
    print("✅ Successfully mapped nodes to clusters.")
else:
    print("⚠️ Warning: No nodes found to cluster.")
    node_cluster_mapping = {}

# Example of checking the result
# for node, cluster in list(node_cluster_mapping.items())[:5]:
#     print(f"Node: {node} is in Cluster: {cluster}")

📊 Clustering 2 nodes into 2 thematic groups...
✅ Successfully mapped nodes to clusters.


### 🏷️ Robust Semantic Clustering

We have implemented a dynamic clustering logic to prevent mathematical errors during the community detection phase.

#### 🛠 Logic Update:
* **Dynamic Centroid Scaling**: The number of clusters (`n_clusters`) now automatically scales based on the unique entity count using `min(5, num_nodes)`.
* **Initialization Safety**: Added checks to handle empty or minimal graphs, ensuring the pipeline doesn't crash during the testing phase.
* **Deterministic Results**: Integrated `random_state` to ensure visual and logical consistency across different sessions.

#### 📍 Practical Result:
This ensures that the **Knowledge Graph** visualization and retrieval logic remain stable whether we are analyzing a simple two-party dispute or a multi-defendant litigation.

In [12]:
from llama_index.core import Settings

def find_similar_nodes_and_edges(query, df, top_n=2):
    """
    Finds the most relevant legal relationships in the graph by comparing 
    the query meaning against the combined meaning of nodes and edges.
    """
    print(f" Searching graph for: {query}")
    
    # 1. 🔥 THE FIX: Use your local BAAI model from Settings instead of OpenAI
    # This ensures the query math matches the graph math
    query_embedding = Settings.embed_model.get_query_embedding(query)
    
    # 2. Vector Comparison
    # We compare the query against a concatenation of the subject, object, and relationship
    df['similarity'] = df.apply(lambda row: cosine_similarity(
        [row['node_1_embedding']] + [row['node_2_embedding']] + [row['edge_embedding']],
        [query_embedding])[0][0], axis=1)

    # 3. Retrieve the winners
    # Picks the top_n (default 2) highest similarity scores
    similar_nodes_df = df.nlargest(top_n, 'similarity')
    
    # Return specific columns including chunk_id for text evidence retrieval
    return similar_nodes_df[['node_1', 'node_2', 'edge', 'chunk_id', 'similarity']]

### 🎯 Semantic Graph Retrieval (BAAI Optimized)

This function implements the primary retrieval logic for our GraphRAG, now optimized for our local embedding infrastructure.

#### 🛠 Key Enhancements:
* **Model Alignment**: Switched from `OpenAIEmbeddings` to `Settings.embed_model` to maintain vector consistency with our BAAI-encoded Knowledge Graph.
* **Feature Fusion**: Performs similarity matching against a concatenated vector of (Subject + Object + Relationship), ensuring the "meaning" of the link is prioritized over single words.
* **Evidence Tracing**: Returns `chunk_id` alongside the similarity score, creating a direct lineage between graph-based conclusions and raw textual evidence in the `lawsuit.txt`.

#### 📍 Practical Application:
This allows the system to identify that a query about **"Delayed Payments"** is semantically identical to a graph edge labeled **"Unpaid Invoices,"** facilitating high-precision legal reasoning.

In [13]:
# function to get text_chunks from chunk Id based on similar nodes with cosine similarity
def get_text_chunks(chunk_ids, chunks_dataframe):
    # Retrieve rows where chunk_id is in the given set of chunk_ids
    relevant_rows = chunks_dataframe[chunks_dataframe['chunk_id'].isin(chunk_ids)]

    # Filter out chunks containing specific strings
    filtered_chunks = relevant_rows[~relevant_rows['text'].str.contains("chunk contextual proximity|global contextual proximity", regex=True)]

    return filtered_chunks['text'].tolist()

# function to extract cosine scores for vector searched nodes based on query
def extract_cosine_scores(similar_nodes_df):
    cosine_scores = {}
    for _, row in similar_nodes_df.iterrows():
        nodes = [row['node_1'], row['node_2']]
        for node in nodes:
            cosine_scores[node] = max(row['similarity'], cosine_scores.get(node, 0))
    return cosine_scores

# function to apply Dijkstra on all node pairs from vector searched nodes
def find_all_shortest_paths(graph, nodes):
    all_paths_str = []

    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            try:
                path = nx.shortest_path(graph, source=nodes[i], target=nodes[j], weight='weight')
                path_str = construct_path_string(path, graph)
                all_paths_str.append(path_str)
            except nx.NetworkXNoPath:
                continue  # If there's no path between a pair of nodes, just skip

    return all_paths_str

# function to make path_string for Dijkstra-paths
def construct_path_string(path, graph):
    path_str = ''
    for i in range(len(path)):
        node = path[i]
        if i > 0:
            prev_node = path[i - 1]
            edge_data = graph.get_edge_data(prev_node, node)
            edge_description = edge_data.get('title', 'Unnamed Edge')
            path_str += f" - {edge_description} - "
        path_str += f"{node}"
    return path_str

# function to apply traveling salesman problem (variation, no circle) on vector-searched nodes
def find_tsp_path(graph, start_node, nodes):
    path = [start_node]
    current_node = start_node
    visited = set([start_node])

    while len(visited) < len(nodes):
        neighbors = [(n, graph[current_node][n]['weight']) for n in graph.neighbors(current_node) if n not in visited]
        if not neighbors:
            break  # No unvisited neighbors, break the loop

        # Sort neighbors by weight
        neighbors.sort(key=lambda x: x[1])

        # Choose the nearest neighbor
        next_node = neighbors[0][0]
        path.append(next_node)
        visited.add(next_node)
        current_node = next_node

    return path

### 🧠 Advanced Graph Reasoning & Context Synthesis

This section contains the "Advanced" logic that separates GraphRAG from standard search engines. Instead of simple keyword matching, we use graph topology and statistical weighting to find the most relevant legal evidence.

#### 🛠 Key Algorithms:
| Function | Purpose | Legal Application |
| :--- | :--- | :--- |
| **Dijkstra Pathfinding** | Finding direct links | Tracing the chain of causation between a defendant and a loss. |
| **TSP Variation** | Logical sequencing | Organizing multiple pieces of evidence into a coherent narrative. |
| **Hybrid Scoring** | Combined Importance | Balancing semantic relevance (Similarity) with structural importance (PageRank). |

#### 📈 Mathematical Strategy:
We normalize all scores (Similarity, TF-IDF, and Centrality) using a `MinMaxScaler`. This prevents one metric from "overpowering" the others, ensuring our **Senior Law Consultant** receives a balanced view of the most significant entities in the `lawsuit.txt`.

#### 📍 Retrieval Workflow:
1. **Extract Facts**: Break the user query into standalone legal points.
2. **Graph Search**: Find the top-scoring nodes for each fact.
3. **Trace Paths**: Find the relationships (Edges) connecting those nodes.
4. **Fetch Chunks**: Pull the original text sentences using `chunk_id`.
5. **Synthesize**: Combine all data into a final prompt for the LLM.

In [14]:
# function to get context_triplets from topic clustering, top n results based on weight
def get_context_triplets(graph, context_nodes, top_n):
    context_triplets = []

    for u, v, data in graph.edges(data=True):
        if u in context_nodes and v in context_nodes:
            edge_description = data.get('title', 'Unnamed Edge')
            edge_weight = data.get('weight', 0)
            triplet = (u, edge_description, v, edge_weight)
            context_triplets.append(triplet)

    # Sort the triplets based on weight and get the top-n results
    context_triplets.sort(key=lambda x: x[3], reverse=True)  # Sorting by edge weight
    return context_triplets[:top_n]

# function to get context_triplets from topic clustering, no weights, all of them
def get_context_triplets_no_weights(graph, context_nodes):
    context_triplets = []

    for u, v, data in graph.edges(data=True):
        if u in context_nodes and v in context_nodes:
            edge_description = data.get('title', 'Unnamed Edge')
            triplet = (u, edge_description, v)
            context_triplets.append(triplet)

    return context_triplets

# function for similarity_search for text
def find_similar_text_chunks(query, vectorstore, parent_child_dict, k=5):
    print(f"Searching for similar chunks to: {query}")
    similar_chunks = vectorstore.similarity_search_with_score(query, k)
    print(f"Found {len(similar_chunks)} similar chunks")

    parent_chunks = set()  
    for doc, score in similar_chunks:
        parent = parent_child_dict[doc.page_content]
        parent_chunks.add(parent) 

    return list(parent_chunks)  # Convert set back to list before returning

# function to apply custom algo to find top_nodes
def generate_context(graph, df_graph, query, tf_idf_scores, top_n=5):
    # Initialize the scaler
    scaler = MinMaxScaler()
    combined_scores = {}  # Initialize the dictionary

    #   Centrality Measures: Highlights structurally significant nodes in the graph with PageRank

    # Calculate centrality measures in the graph
    centrality = nx.pagerank(graph)  # or other centrality measures
    # Normalize the centrality scores
    centrality_values = list(centrality.values())
    centrality_normalized = scaler.fit_transform(np.array(centrality_values).reshape(-1, 1)).flatten()

    # Map back the normalized scores to nodes
    normalized_centrality = dict(zip(centrality.keys(), centrality_normalized))

    #   TF-IDF Scores: Signifies the uniqueness and importance of nodes in various contexts

    tf_idf_values = list(single_tf_idf_scores.values())
    tf_idf_normalized = scaler.fit_transform(np.array(tf_idf_values).reshape(-1, 1)).flatten()

    # Map back the normalized scores to nodes
    normalized_tf_idf_scores = dict(zip(single_tf_idf_scores.keys(), tf_idf_normalized))

    #    Cosine Similarity: Determines the relevance of nodes to the query

    # Find similar nodes and edges
    similar_nodes_df = find_similar_nodes_and_edges(query, df_graph, top_n=10)

    # Extract cosine similarity scores for individual nodes
    cosine_scores = extract_cosine_scores(similar_nodes_df)

    #   combined scores for top nodes
    for node in graph.nodes():
        cosine_score = cosine_scores.get(node, 0)
        tf_idf_score = normalized_tf_idf_scores.get(node, 0)
        centrality_score = normalized_centrality.get(node, 0)
        combined_scores[node] = cosine_score + tf_idf_score + centrality_score

    # Select top scoring nodes
    top_nodes = sorted(combined_scores, key=combined_scores.get, reverse=True)[:top_n]
    return top_nodes


### 🧠 Advanced Context Synthesis & Hybrid Ranking

This section implements the sophisticated retrieval logic that identifies the most significant "evidentiary anchors" in our legal dataset.

#### 🛠 1. Relational Extraction (`get_context_triplets`)
We use these functions to extract structured "Subject-Predicate-Object" chains from our thematic clusters. By prioritizing edges with higher weights, we ensure the LLM receives the most well-documented relationships first.

#### 📖 2. Parent-Child Text Retrieval (`find_similar_text_chunks`)
To prevent the loss of legal nuance, we implement a **Parent-Child mapping**.
* **Child Retrieval**: High-precision vector search on small text segments.
* **Parent Expansion**: Automatic retrieval of the original paragraph/section to provide the LLM with the full legal context surrounding a specific fact.

#### ⚖️ 3. Multi-Factor Scoring Engine (`generate_context`)
We implement a **Hybrid Ranking** algorithm that calculates node importance through three distinct lenses:
1. **Structural Centrality (PageRank)**: How many other legal facts rely on this node?
2. **Contextual Uniqueness (TF-IDF)**: How specific is this node to the current legal argument?
3. **Semantic Alignment (Cosine Similarity)**: How directly does this node answer the user's question?

This "Weighted Voting" system ensures that our **Senior Law Consultant** focuses on evidence that is structurally sound, statistically significant, and semantically relevant.

In [15]:
# function to get text chunks for top nodes
def get_text_chunks_for_top_nodes(top_nodes, df_graph, chunks_dataframe):
    # Retrieve chunk IDs associated with top nodes
    chunk_ids = set()
    for node in top_nodes:
        rows = df_graph[(df_graph['node_1'].isin(node)) | (df_graph['node_2'].isin(node))]
        for _, row in rows.iterrows():
            chunk_ids.update(row['chunk_id'].split(','))

    # Fetch corresponding text chunks
    return get_text_chunks(chunk_ids, chunks_dataframe)

### 📖 Function: `get_text_chunks_for_top_nodes` (Targeted Evidence Retrieval)

This function automates the retrieval of raw textual evidence for the most significant entities identified by our hybrid ranking engine.

#### 🛠 Technical Mechanics:
* **Bidirectional Search**: Scans both `node_1` and `node_2` columns in the `df_graph` to ensure full coverage of an entity's involvement in the case.
* **Deduplicated Indexing**: Utilizes a Python `set` to aggregate unique `chunk_id` references, optimizing context window usage by preventing redundant information.
* **Traceable Retrieval**: Bridges the gap between the abstract Knowledge Graph and the verbatim source text in the `lawsuit.txt`.

#### 📍 Practical Application:
If the system determines that **"Article 12 of the Agreement"** is a top-priority node, this function will instantly find every paragraph in the corpus that mentions Article 12, allowing the LLM to provide a response based on the actual contract language.

In [16]:
from llama_index.core import Settings
from llama_index.core.llms import ChatMessage

def generate_fact_extract(query):
    # Use Settings.llm to ensure the object is found
    llm_engine = Settings.llm 
    
    system_prompt = system_prompts.get('system_prompt_fact_extract', "Extract legal facts.")
    
    try:
        messages = [
            ChatMessage(role="system", content=system_prompt),
            ChatMessage(role="user", content=query)
        ]
        
        response = llm_engine.chat(messages=messages)
        return response.message.content
        
    except Exception as e:
        print(f"❌ Error during fact extraction: {e}")
        # Return a valid list format as a fallback so the next step doesn't crash
        return "['sonstiges', 'General query']"

### ✂️ Function: `generate_fact_extract` (Query Deconstruction)

This function serves as the **Preprocessing Layer** of our RAG pipeline. It utilizes **Groq-accelerated Llama 3.3** to transform unstructured user input into structured search parameters.

#### 🛠 Technical Mechanics:
* **Atomic Fact Extraction**: Breaks complex legal inquiries into a list of singular entities and legal concepts.
* **Format Enforcement**: Utilizes a specialized system prompt (`system_prompt_fact_extract`) to ensure the LLM returns a strictly formatted Python list.
* **Safety Parsing**: Employs `ast.literal_eval` to safely evaluate the string output into a machine-readable format for downstream graph traversal.

#### 📍 Practical Benefit:
This deconstruction allows our **Semantic Search** to target multiple specific regions of the Knowledge Graph simultaneously. Instead of searching for one long sentence, the system searches for the Subject, the Action, and the Object separately, significantly increasing retrieval accuracy.

In [17]:
#function to determine the system_prompt based on the first entry in the list returned from fact_extract
def get_system_prompt(fact_extract):
    print("start get_system_prompt")
    query_type = fact_extract[0]
    print(fact_extract[0])
    print("query_type: ", query_type)
    query_type = "system_prompt_" + query_type.lower()
    if query_type in system_prompts:
        return system_prompts[query_type]
    else:
        return system_prompts['system_prompt_sonstiges']

### 🚦 Function: `get_system_prompt` (Dynamic Agent Routing)

This function implements **Conditional Instruction Loading**, allowing the AI to pivot its reasoning style based on the classified intent of the user.

#### 🛠 Technical Mechanics:
* **Intent Mapping**: Leverages the first element of the `fact_extract` list as a routing key.
* **String Normalization**: Standardizes query types to match the filenames in the `system_prompts/` directory.
* **Fail-Safe Mechanism**: Implements a "Sonstiges" (Miscellaneous) fallback to ensure the system remains functional even if the intent classifier produces an unexpected label.

#### 📍 Practical Benefit:
This architecture enables a **Multi-Agent feel** within a single LLM. By switching the system prompt, we can force the LLM to switch from a "Document Summarizer" role to a "Strict Legal Compliance Auditor" role without changing the underlying code.

In [18]:
from llama_index.core import Settings

# function to Generate Response with Groq (previously OpenAI)
def generate_response(query, fact_extract):

    # ==============================
    # find similar_nodes with cos similarity
    # ==============================

    # Using list comprehension
    similar_nodes = [find_similar_nodes_and_edges(query, df_graph, top_n=2) for query in fact_extract]

    # Format graph information of similar nodes
    graph_info = "\n".join(
        [f"Node: {row['node_1']} ist mit Node: {row['node_2']} verbunden, durch die Edge: '{row['edge']}'" 
        for df in similar_nodes for _, row in df.iterrows()]
    )

    # Retrieve unique chunk_ids from the similar nodes
    chunk_ids = set()
    for df in similar_nodes:
        for _, row in df.iterrows():
            chunk_ids.update(row['chunk_id'].split(','))

    # Fetch the corresponding text chunks with the filtering applied
    cosine_nodes_chunks = get_text_chunks(chunk_ids, chunks_dataframe)
    # Outside the f-string
    cosine_nodes_chunks_str = '\n'.join(cosine_nodes_chunks)

    # ==============================
    # get TSP(Variant) between vector searched nodes
    # ==============================

    # Get the list of similar nodes
    similar_nodes_list = [row['node_1'] for df in similar_nodes for _, row in df.iterrows()]

    # Find the TSP path
    # Using the first similar node as start point
    tsp_path = find_tsp_path(graph, similar_nodes_list[0], similar_nodes_list)

    # Construct the string representation of the TSP path
    tsp_path_str = ''
    for i in range(len(tsp_path)):
        node = tsp_path[i]
        if i > 0:
            prev_node = tsp_path[i - 1]
            edge_data = graph.get_edge_data(prev_node, node)
            edge_description = edge_data['title']  # Adjust key as needed
            tsp_path_str += f" - {edge_description} - "
        tsp_path_str += f"{node}"

    # ==============================
    # get shortest_paths between all similar_nodes pairs (Dijkstra)
    # ==============================
    # Find all shortest paths
    shortest_paths_strings = find_all_shortest_paths(graph, similar_nodes_list)

    # ==============================
    # get contextual_ nodes with tf-idf balanced with weights for similar_nodes
    # ==============================

    # Initialize a string to hold contextual node information
    contextual_nodes_info = ""

    for df in similar_nodes:
        for index, row in df.iterrows():
            node = row['node_1']
            # Retrieve the top neighbors based on edge weight
            top_nodes_edges = get_top_contextual_nodes_edges(graph, node, top_n=2)

            contextual_nodes_info += f"Node: {node}\n"
            for neighbor, weight in top_nodes_edges:
                tf_idf_score = tf_idf_scores.get(node, {}).get(neighbor, 0)
                # Combine the scores: Adjust the '0.5' factors to tweak the balance
                combined_score = 0.5 * tf_idf_score + 0.4 * weight

                edge_data = graph.get_edge_data(node, neighbor) or {}
                edge_title = edge_data.get('title', '')
                contextual_info = f" - Contextual Neighbor: {neighbor}, Edge: {edge_title}, Combined Score: {combined_score}\n"
                contextual_nodes_info += contextual_info

    # ==============================
    # topic clustering of nodes
    # ==============================

    # Find the clusters of the similar nodes
    similar_nodes_clusters = {node_cluster_mapping[node] for node in similar_nodes_list if node in node_cluster_mapping}

    # Get all nodes from these clusters
    context_nodes = [node for node, cluster in node_cluster_mapping.items() if cluster in similar_nodes_clusters]

    # Get the top-n topic clustered triplets (weighted)
    context_triplets = get_context_triplets(graph, context_nodes, top_n=3)
    
    # Format these triplets as a string
    topic_triplets_str = "\n".join([f"{u} - {edge} - {v}" for u, edge, v, _ in context_triplets])

    # ==============================
    # custom algo for top nodes
    # ==============================

    # Generate context using the custom algorithm
    top_nodes = [generate_context(graph, df_graph, query, tf_idf_scores, top_n=1) for query in fact_extract]
    # Flattening the list of lists
    top_nodes = [node for sublist in top_nodes for node in sublist]

    # Extract node-edge-node triplets for these top nodes
    top_nodes_triplets = get_context_triplets(graph, top_nodes, top_n = 2)
    top_nodes_triplets_str = "\n".join([f"{u} - {edge} - {v}" for u, edge, v, _ in top_nodes_triplets])

    # ==============================
    # get contextual_ nodes with tf-idf balanced with weights for top_nodes
    # ==============================

    contextual_nodes_info_top_nodes = ""

    for node in top_nodes:
        top_nodes_edges = get_top_contextual_nodes_edges(graph, node, top_n=2)
        contextual_nodes_info_top_nodes += f"Node: {node}\n"
        for neighbor, weight in top_nodes_edges:
            tf_idf_score = tf_idf_scores.get(node, {}).get(neighbor, 0)
            combined_score = 0.5 * tf_idf_score + 0.4 * weight
            edge_data = graph.get_edge_data(node, neighbor) or {}
            edge_title = edge_data.get('title', '')
            contextual_info = f" - Contextual Neighbor: {neighbor}, Edge: {edge_title}, Combined Score: {combined_score}\n"
            contextual_nodes_info_top_nodes += contextual_info

    # ==============================
    # get text chunks with cos similarity
    # ==============================

    # Get referenced text chunks using local vectorstore
    similar_chunk_for_query = find_similar_text_chunks(query, text_vectorstore, k=5, parent_child_dict=parent_child_dict)
    similar_chunks = [find_similar_text_chunks(q, text_vectorstore, k=5, parent_child_dict=parent_child_dict) for q in fact_extract]

    # Format text references
    references_query = "\n".join([f"{content}" for content in similar_chunk_for_query])
    # Flatten similar_chunks for formatting
    flattened_references = [item for sublist in similar_chunks for item in sublist]
    references = "\n".join([f"{content}" for content in flattened_references])

    # ==============================
    # combine all context
    # ==============================

    combined_context = (
                        f"User Query:\n {query}"
                        f"\n Facts:\n {fact_extract}"
                        f"\n\n Korpus aus Knowledge Graph (A) und referenzierten Textstellen (B): \n"
                        f"A. Kontext vom Knowledge Graphen - \n"
                        f"1. cosine similarity Vector-searched Nodes und Edges vom Knowledge Graphen :\n{graph_info}\n\n"
                        f"2. Kontextuelle Nachbarn der vorherigen Nodes :\n{contextual_nodes_info}\n\n"
                        f"3. Kürzester Weg durch den Graphen, der alle vector-searched nodes verbindet: \n{tsp_path_str}\n\n"
                        f"4. Topic Clusters im graphen basierend auf query:\n{topic_triplets_str}\n\n"
                        f"5. weitere top_nodes für das query:\n{top_nodes_triplets_str}\n\n"
                        f"6. kontextuelle Nachbarn der weiteren top_nodes:\n {contextual_nodes_info_top_nodes}\n\n"
                        f"B. Kontext durch referenzierte Textstellen aus dem Corpus:\n{references_query}\n{references}\n"
                        )

    # ==============================
    # make LLM call (Migrated to Groq)
    # ==============================

    system_prompt = get_system_prompt(fact_extract)
    print("🚀 Starting Groq LLM Call...")
    
    # Use Settings.llm (configured as Groq) instead of OpenAI client
    llm = Settings.llm
    
    # Generate response using chat interface
    response_obj = llm.chat(
        messages=[
            {"role": "system", "content": f"{system_prompt}"},
            {"role": "user", "content": combined_context}
        ]
    )

    response = response_obj.message.content
    print("✅ Got response from Groq.")

    # Combine context and response for the final output
    full_interaction = (
                        f"\n\nUser Query:\n {query}"
                        f"\n Facts:\n {fact_extract}"
                        f"\n Response des LLM:\n {response}"
                        f"\n\n Korpus aus Knowledge Graph (A) und referenzierten Textstellen aus dem Grundkorpus (B): \n"
                        f"A. Kontext vom Knowledge Graphen - \n"
                        f"1. cosine similarity Vector-searched Nodes und Edges vom Knowledge Graphen :\n{graph_info}\n\n"
                        f"2. Kontextuelle Nachbarn der vorherigen Nodes :\n{contextual_nodes_info}\n\n"
                        f"3. Kürzester Weg durch den Graphen, der alle vector-searched nodes verbindet: \n{tsp_path_str}\n\n"
                        f"4. Topic Clusters im graphen basierend auf query:\n{topic_triplets_str}\n\n"
                        f"5. weitere top_nodes für das query:\n{top_nodes_triplets_str}\n\n"
                        f"6. kontextuelle Nachbarn der weiteren top_nodes:\n {contextual_nodes_info_top_nodes}\n\n"
                        f"B. Kontext durch referenzierte Textstellen aus dem Corpus:\n{references_query}\n{references}\n"
                        )
    return full_interaction

### 🏛️ The Master Orchestration Engine: `generate_response`

This function is the central brain of our Legal GraphRAG. It synthesizes structured graph data with unstructured textual evidence to provide comprehensive legal answers.

#### 🛠 Integrated Pipeline:
1. **Multi-Fact Retrieval**: Processes each extracted fact from the query to find its semantic match in the Knowledge Graph.
2. **Structural Synthesis**: Uses **TSP (Traveling Salesman) logic** to find a narrative path between facts and **Dijkstra** to find the shortest logical connection between entities.
3. **Thematic Clustering**: Identifies the "Topic Community" of the retrieved nodes to include relevant secondary context.
4. **Hybrid Context Construction**: Combines specialized Knowledge Graph triplets (A) with verbatim text references from the corpus (B).

#### 🚀 Groq Acceleration:
The function has been migrated to utilize the **Groq Llama-3.3-70b** model via the LlamaIndex `Settings` object, ensuring low-latency inference while maintaining strict adherence to the provided legal system prompts.

#### 📍 Output Strategy:
The final return value is a **Full Interaction Trace**, which includes the LLM's response followed by a detailed breakdown of all evidence used (Paths, Clusters, and Text Chunks). This provides the **Senior Law Consultant** with a "Checkable" output for legal verification.

In [19]:
import os
import glob
import re
import json
from langchain_community.vectorstores import Chroma
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core import Document, Settings
from typing import List

# 1. Custom Wrapper (Unchanged logic, added type safety)
class LlamaIndexWrapper:
    def __init__(self, llama_index_embed_model):
        self._model = llama_index_embed_model
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._model.get_text_embedding_batch(texts)
    def embed_query(self, text: str) -> List[float]:
        return self._model.get_query_embedding(text)

# 2. Configuration & Paths (Check these carefully!)
data_input_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data"
text_vectorstore_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vectorstores\text"
parent_child_dict_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vectorstores\parent_child_dict.json"

# 3. Initialize Model
langchain_embed_model = LlamaIndexWrapper(Settings.embed_model)

# 4. Setup the Semantic Splitter
splitter = SemanticSplitterNodeParser(
    buffer_size=1, 
    breakpoint_percentile_threshold=95, 
    embed_model=Settings.embed_model
)

# 5. Execution Logic
if os.path.exists(text_vectorstore_path) and os.path.exists(parent_child_dict_path):
    print("📂 SUCCESS: Loading existing vectorstore and dictionary...")
    text_vectorstore = Chroma(
        persist_directory=text_vectorstore_path, 
        embedding_function=langchain_embed_model
    )
    with open(parent_child_dict_path, 'r', encoding="utf-8-sig") as file:
        parent_child_dict = json.load(file)
else:
    print("🆕 STARTING INGESTION: Processing local text files...")
    os.makedirs(os.path.dirname(text_vectorstore_path), exist_ok=True)

    def hi_text_splitter(directory):
        chunks_with_meta = []
        files = glob.glob(f"{directory}/*.txt")
        print(f"🔎 Found {len(files)} .txt files in the data directory.")
        
        for file_path in files:
            file_name = os.path.basename(file_path)
            with open(file_path, 'r', encoding='utf-8-sig') as file:
                text = file.read()
                hi_indices = [match.start() for match in re.finditer(r'HI\d+', text)]
                
                # Split based on HI indicators or keep whole
                if hi_indices:
                    file_chunks = []
                    file_chunks.append(text[:hi_indices[0]])
                    for start, end in zip(hi_indices, hi_indices[1:] + [None]):
                        file_chunks.append(text[start:end])
                    for c in file_chunks: chunks_with_meta.append({"text": c, "source": file_name})
                else:
                    chunks_with_meta.append({"text": text, "source": file_name})
        return chunks_with_meta

    # Process files
    raw_parent_data = hi_text_splitter(data_input_path)
    
    if not raw_parent_data:
        print("🛑 CRITICAL: No text found in lawsuit.txt! Check your file path.")
    
    all_child_chunks = []
    parent_child_dict = {}

    print("🧠 Applying Semantic Splitting (this may take a moment)...")
    for item in raw_parent_data:
        parent_text = item["text"]
        source_file = item["source"]
        
        # Split parent text into semantically relevant child nodes
        nodes = splitter.get_nodes_from_documents([Document(text=parent_text, metadata={"file_name": source_file})])
        
        for node in nodes:
            child_text = node.get_content()
            all_child_chunks.append(child_text)
            # Map child back to parent for contextual retrieval
            parent_child_dict[child_text] = parent_text

    print(f"🗳️ Vectorizing {len(all_child_chunks)} chunks with BAAI/bge-small-en-v1.5...")
    text_vectorstore = Chroma.from_texts(
        texts=all_child_chunks,
        embedding=langchain_embed_model,
        persist_directory=text_vectorstore_path
    )

    # Save the lookup dictionary
    with open(parent_child_dict_path, 'w', encoding='utf-8-sig') as file:
        json.dump(parent_child_dict, file, ensure_ascii=False, indent=4)

    print(f"✅ Ingestion complete. {len(all_child_chunks)} chunks stored.")

📂 SUCCESS: Loading existing vectorstore and dictionary...


C:\Users\user\AppData\Local\Temp\ipykernel_13700\3887817652.py:37: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  text_vectorstore = Chroma(


### 🏛️ Document Ingestion & persistence Pipeline

This module manages the transformation of raw legal text into a searchable, persistent vector database.

#### 🛠 Key Components:
1. **Adapter Pattern (`LlamaIndexWrapper`)**: 
   - Bridges the interface gap between `LlamaIndex` embedding objects and `LangChain` vector stores.
   - Maps `get_text_embedding_batch` to `embed_documents`.

2. **Structural Partitioning (`hi_text_splitter`)**: 
   - Uses Regular Expressions (`HI\d+`) to identify natural boundaries in legal/transactional documents.
   - Preserves the integrity of major sections before granular chunking.

3. **Semantic Refinement**:
   - Employs the `SemanticSplitterNodeParser` to identify thematic breakpoints.
   - **Threshold**: 95th percentile (ensures chunks are only created at significant topic shifts).

4. **Hierarchical Metadata Mapping**:
   - Implements a **Parent-Child** relationship model.
   - Search is performed on "Child" nodes (semantic snippets), while "Parent" nodes (full HI sections) are retrieved for the LLM to ensure maximum legal context.

#### 📍 Operational Workflow:
- **Phase 1**: Check for local assets at `vectorstores/`.
- **Phase 2**: If null, parse source `.txt` files via Regex markers.
- **Phase 3**: Generate semantic nodes and populate the relational dictionary.
- **Phase 4**: Commit vectors to `Chroma` and serialize the dictionary to `JSON`.

In [20]:
import os
from datetime import datetime

# 1. Knowledge Graph (KG) Vectorstore Persistence
# This ensures we don't re-calculate embeddings for nodes/edges every time
kg_vectorstore_path = r"C:\Users\user\Desktop\RAG Projects\Legal RAG 1\vectorstores\kg"

# Use the LlamaIndexWrapper (langchain_embed_model) defined in previous steps
if os.path.exists(kg_vectorstore_path):
    print("📂 Loading existing vectorstore for Knowledge Graph...")
    kg_vectorstore = Chroma(
        persist_directory=kg_vectorstore_path, 
        embedding_function=langchain_embed_model
    )
else:
    print("🆕 Creating new vectorstore for Knowledge Graph...")
    # Extract all unique entities and relationships from your graph dataframe
    kg_texts = df_graph['node_1'].tolist() + df_graph['node_2'].tolist() + df_graph['edge'].tolist()
    
    kg_vectorstore = Chroma.from_texts(
        texts=kg_texts,
        embedding=langchain_embed_model,
        persist_directory=kg_vectorstore_path
    )
    print("✅ KG Vectorstore initialized.")

# 2. Main Interaction Loop
chat_history = []
query = None

print("\n⚖️ Legal Document Intelligence System Ready (Type 'exit' to quit)")
print("-" * 50)

while True:
    query = input("Prompt: ")
    if query.lower() in ['quit', 'q', 'exit']:
        break
    
    # Step A: Deconstruct the query into atomic facts using Groq
    fact_extract = generate_fact_extract(query)
    print(f"🔍 Extracted Facts: {fact_extract}")
    
    # Step B: Generate the full RAG response (Graph + Text + Groq)
    full_interaction = generate_response(query, fact_extract)
    
    # Display only the LLM Response part to the user for clarity
    print("\n" + "="*30 + " RESPONSE " + "="*30)
    print(full_interaction)
    print("="*70 + "\n")
    
    # Store the interaction for the history file
    chat_history.append(full_interaction)

# 3. Session Archiving
# Save the entire session to a text file for future legal auditing
history_filename = './data_output/history/chat_history.txt'
os.makedirs(os.path.dirname(history_filename), exist_ok=True)

with open(history_filename, 'a', encoding='utf-8') as file:
    file.write("\n" + "#" * 80 + "\n")
    file.write(f"SESSION DATE: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    file.write("\n\n".join(chat_history) + "\n\n")

print(f"💾 Session history saved to {history_filename}")

📂 Loading existing vectorstore for Knowledge Graph...

⚖️ Legal Document Intelligence System Ready (Type 'exit' to quit)
--------------------------------------------------
❌ Error during fact extraction: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
🔍 Extracted Facts: ['sonstiges', 'General query']
 Searching graph for: [
 Searching graph for: '
 Searching graph for: s
 Searching graph for: o
 Searching graph for: n
 Searching graph for: s
 Searching graph for: t
 Searching graph for: i
 Searching graph for: g
 Searching graph for: e
 Searching graph for: s
 Searching graph for: '
 Searching graph for: ,
 Searching graph for:  
 Searching graph for: '
 Searching graph for: G
 Searching graph for: e
 Searching graph for: n
 Searching graph for: e
 Searching graph for: r
 Searching graph for: a
 Searching graph for: l
 Searching graph for:  
 Searching graph for: q
 Searching graph for: u
 Searching graph for: e
 S

KeyError: 'system_prompt_sonstiges'

### Neo4j Graph Database

In [ ]:
# 1. Use the store object you already initialized
# This sends the Cypher command to delete everything
try:
    print("🧹 Clearing the Neo4j database...")
    
    # 'DETACH DELETE' removes nodes AND their connected relationships
    property_graph_store.structured_query("MATCH (n) DETACH DELETE n")
    
    print("✅ Database cleared successfully.")
except Exception as e:
    print(f"❌ ERROR: Could not clear database: {e}")

🧹 Clearing the Neo4j database...
❌ ERROR: Could not clear database: name 'property_graph_store' is not defined


In [ ]:
import os
import json
import networkx as nx
from py2neo import Graph, Node, Relationship
from dotenv import load_dotenv

# 1. Load variables and sanitize URI for py2neo
load_dotenv(override=True)
RAW_URI = os.getenv('NEO4J_URI') or "bolt://localhost:7687"
SAFE_URI = RAW_URI.replace("neo4j://", "bolt://")
USER = os.getenv('NEO4J_USERNAME')
PASS = os.getenv('NEO4J_PASSWORD')

# Temporarily hide ENV variable to prevent py2neo protocol crash
if "NEO4J_URI" in os.environ:
    del os.environ["NEO4J_URI"]

try:
    print(f"📡 Connecting to Neo4j via: {SAFE_URI}")
    neo4j_db = Graph(SAFE_URI, auth=(USER, PASS))
    neo4j_db.run("MATCH (n) RETURN count(n) LIMIT 1")
    print("✅ Connection Successful.")
except Exception as e:
    print(f"❌ Connection Failed: {e}")

# Restore for other components
os.environ["NEO4J_URI"] = RAW_URI

def migrate_to_neo4j(graph_obj):
    print("🧹 Wiping old data...")
    neo4j_db.delete_all()
    print(f"🧬 Migrating {len(graph_obj.nodes)} entities...")
    for node_id in graph_obj.nodes:
        vector = langchain_embed_model.embed_query(str(node_id))
        node_obj = Node("Entity", "Chunk", id=str(node_id), name=str(node_id), text=str(node_id), embedding=vector)
        neo4j_db.create(node_obj)
    
    for u, v, data in graph_obj.edges(data=True):
        source_node = neo4j_db.nodes.match("Entity", id=str(u)).first()
        target_node = neo4j_db.nodes.match("Entity", id=str(v)).first()
        if source_node and target_node:
            rel_type = str(data.get('title', 'RELATED_TO')).replace(" ", "_").upper()
            rel = Relationship(source_node, rel_type, target_node, **data)
            neo4j_db.create(rel)
    print("✅ Migration Successful.")

# 2. THE JSON FIX: Check for empty files before loading
json_path = r'C:\Users\user\Desktop\RAG Projects\Legal RAG 1\data_output\graph_data.json'

if not os.path.exists(json_path):
    print(f"❌ ERROR: File not found at {json_path}")
elif os.path.getsize(json_path) == 0:
    print(f"🛑 ERROR: The file '{json_path}' is EMPTY (0 bytes).")
    print("👉 FIX: You must re-run the 'Entity Extraction' cell to generate the graph data first.")
else:
    try:
        with open(json_path, 'r', encoding='utf-8-sig') as f:
            data = json.load(f)
            nx_graph = nx.node_link_graph(data)
            migrate_to_neo4j(nx_graph)
    except json.JSONDecodeError:
        print("❌ ERROR: JSON is malformed. Delete 'graph_data.json' and re-run the extraction cell.")

📡 Connecting to Neo4j via: bolt://localhost:7687
❌ Connection Failed: Unknown protocol 'neo4j'
🧹 Wiping old data...


c:\Users\user\AppData\Local\Programs\Python\Python310\lib\site-packages\networkx\readwrite\json_graph\node_link.py:287: FutureWarning: 
The default value will be changed to `edges="edges" in NetworkX 3.6.

To make this warning go away, explicitly set the edges kwarg, e.g.:

  nx.node_link_graph(data, edges="links") to preserve current behavior, or
  nx.node_link_graph(data, edges="edges") for forward compatibility.
  warnings.warn(


NameError: name 'neo4j_db' is not defined

### 🏛️ Neo4j Relational Migration & Vector Integration

This script serves as the bridge between our local Python-based graph and the **Neo4j Graph Database**, enabling high-performance relational queries.

#### 🛠 technical Improvements:
* **Vectorized Properties**: Every node in Neo4j is now enriched with a `BAAI/bge-small-en-v1.5` embedding. This facilitates **Hybrid Graph Retrieval**, where we can search by both structural connection and semantic similarity.
* **Structural Integrity**: Maintains `weight` and `title` metadata across the migration, ensuring that the **Advocatus Facti** can still verify the strength of evidence within the database.
* **Framework Synchronization**: Utilizes the `LlamaIndexWrapper` to ensure the Neo4j vector space is identical to the Chroma vector store.

#### 📍 Operational Result:
The Knowledge Graph is no longer a static file; it is a live, searchable database capable of visualizing complex legal networks, such as the relationship between Constitutional Articles and specific Case Law precedents.

### 🛡️ Neo4j Credential Management

Before executing the migration, ensure your environment variables are configured with valid credentials.

#### 📍 Source Selection:
* **Local (Development)**: Managed via **Neo4j Desktop**. Credentials are set during database creation.
* **Cloud (Production)**: Managed via **Neo4j Aura Console**. Credentials must be downloaded as a `.txt` file during instance setup.

#### 🛠 Security Best Practices:
1. **Never Hardcode**: Always use a `.env` file to store `NEO4J_PASSWORD`.
2. **First Login**: For new local instances, the default password is often `neo4j`. The system will require an immediate reset upon first connection.
3. **URI Protocols**: 
   - Use `bolt://` or `neo4j://` for local non-encrypted connections.
   - Use `neo4j+s://` for encrypted cloud connections (Aura).

### 🛡️ JSON Integrity & Decoding Fix

We have updated the graph loader to handle the `JSONDecodeError: Expecting value` issue.

#### 🛠 Debugging Insights:
* **The Cause**: This error typically indicates an **Empty File** (0 bytes) or an **Encoding Mismatch**. 
* **The Logic**: Added `os.path.getsize` validation to prevent the decoder from attempting to parse null data.
* **Encoding Safety**: Switched to `utf-8-sig` to automatically strip invisible Byte Order Marks (BOM) that certain Windows-based editors insert at the start of files.

#### 📍 Troubleshooting Steps:
1. Re-run the **Graph Generation** cell to ensure `graph_data.json` is populated.
2. Confirm the output path matches: `./data_output/graph_data.json`.
3. Check the file manually to ensure it begins with a valid `{` character.

In [ ]:
import os
from llama_index.core import Settings, PropertyGraphIndex
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.llms.groq import Groq
from dotenv import load_dotenv

# 1. Load Environment Variables
load_dotenv(override=True)

NEO4J_URI = os.getenv('NEO4J_URI') or os.getenv('NEO4J_URL')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')
GROQ_KEY = os.getenv("GROQ_API_KEY")

# 2. Global Model Settings
Settings.llm = Groq(model="llama-3.3-70b-versatile", api_key=GROQ_KEY)
llm = Settings.llm 
embed_model = Settings.embed_model 

try:
    # 3. Initialize the Property Graph Store
    property_graph_store = Neo4jPropertyGraphStore(
        url=NEO4J_URI,           
        username=NEO4J_USERNAME, 
        password=NEO4J_PASSWORD
    )

    # 4. FIXED PRE-FLIGHT CHECK: 
    # Use the property_graph_store directly to run the query
    check_query = "MATCH (n:Entity) RETURN count(n) as count"
    node_count_result = property_graph_store.structured_query(check_query)
    
    # Extracting count safely from the list of dictionaries
    count = node_count_result[0].get('count', 0) if node_count_result else 0
    print(f"📊 DATABASE STATUS: Found {count} nodes with label 'Entity'.")

    # 5. Build Index from existing data
    index = PropertyGraphIndex.from_existing(
        property_graph_store=property_graph_store,
        llm=llm,
        embed_model=embed_model
    )

    # 6. Configure Query Engine
    query_engine = index.as_query_engine(include_text=True, similarity_top_k=10)

    # 7. Execution
    user_query = input("\n⚖️ Enter your legal query: ")
    print(f"🔍 Analyzing: {user_query}...")
    
    response = query_engine.query(user_query)

    print("\n" + "="*20 + " ANALYSIS " + "="*20)
    print(response)
    print("="*50)

except Exception as e:
    print(f"❌ ERROR: {e}")

📊 DATABASE STATUS: Found 2 nodes with label 'Entity'.
🔍 Analyzing: Finds the Entity node for "Smith" (Retrieval)....

==================== ANALYSIS ====================
Empty Response


### 🧠 Knowledge Graph Query Engine (KGQE)

This module serves as the **Natural Language Interface** to our vectorized Neo4j database. It eliminates the need for manual Cypher scripting during the research phase.

#### 🛠 Logic & Components:
* **Translation Layer**: Utilizes **Groq (Llama-3.3-70b)** to interpret user intent and generate syntactically correct Cypher queries. 
* **Graph Store Integration**: Leverages `Neo4jGraphStore` to maintain a persistent connection to the local database instance.
* **Metadata Awareness**: The engine is "Schema-Aware," meaning it understands the `:LegalEntity` labels and `RELATED_TO` relationships we migrated in the previous step.

#### 📍 Operational Workflow:
1. **Schema Retrieval**: The engine fetches the graph schema (Labels, Relationship Types).
2. **Query Synthesis**: The LLM writes a Cypher query targeted at the relevant nodes (e.g., finding the path between a "Constitutional Article" and a "Supreme Court Judgment").
3. **Graph Traversal**: The query is executed, traversing the network to find non-obvious legal connections.
4. **Final Synthesis**: The raw graph results are passed back to the LLM to be formatted into the final legal opinion.